# nb13: Symmetry Descriptors via spglib (pymatgen wrapper)

**Library:** `spglib` (the C library) accessed through pymatgen's `SpacegroupAnalyzer`. Standard tool for crystal symmetry analysis. Already installed if you have pymatgen.

**Why this is physically motivated for Rashba:**
Rashba SOC requires broken inversion symmetry. The space group of a crystal *directly tells you* whether inversion symmetry is present. Compounds with centrosymmetric space groups (like P-1, P2_1/c, Fmmm) have α_R = 0 by symmetry. Non-centrosymmetric groups (like Pm, P3m1, Cm) allow non-zero α_R. So symmetry descriptors should split the dataset cleanly by whether Rashba is even *allowed*.

**What spglib gives us:**
1. Space group number (1-230) and Hermann-Mauguin symbol (e.g., "P3m1")
2. Crystal system (triclinic/monoclinic/.../cubic)
3. Point group (32 possibilities)
4. Whether centrosymmetric (yes/no) — KEY for Rashba
5. Number of symmetry operations (order of point group)
6. Layer group / 2D space group (for 2D materials specifically)
7. Wyckoff positions (which atoms sit at high-symmetry sites)

**Strategy:** Same C6+1 / C6+2 approach as nb9 and nb10. Hard cap 10 features.

**Honest caveat for the report:** if symmetry features improve R² substantially, that's a real finding. If they don't, it's because most of your 99 compounds are already non-centrosymmetric (otherwise their α_R would be 0 and they wouldn't be in the dataset), so symmetry doesn't discriminate within the set.


## Cell 1: Imports & paths

In [9]:
import os
import glob
import pandas as pd

# Adjust if needed
POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')

# Files we want to check for
FILES_TO_CHECK = ['POSCAR', 'POSCAR_std', 'CONTCAR', 'INCAR', 'POTCAR',
                  'KPOINTS', 'vasprun.xml', 'WAVECAR', 'DOSCAR', 'CHGCAR',
                  'OUTCAR', 'lobsterin', 'COHPCAR.lobster', 'ICOHPLIST.lobster']

# Also, what does the INCAR say about LWAVE?
def check_lwave_in_incar(incar_path):
    if not os.path.exists(incar_path):
        return 'INCAR missing'
    try:
        with open(incar_path) as f:
            txt = f.read()
        # Find LWAVE setting
        for line in txt.splitlines():
            line = line.split('#')[0].strip()  # strip comments
            if 'LWAVE' in line.upper():
                return line
        return 'LWAVE not specified (default = FALSE)'
    except Exception as e:
        return f'read error: {e}'


# Find all compound folders
compound_dirs = [d for d in glob.glob(os.path.join(POSCAR_DIR, '*'))
                 if os.path.isdir(d)]
print(f'Found {len(compound_dirs)} compound folders\n')

# Check first 5 folders in detail
print('=' * 80)
print('DETAILED CHECK: first 5 folders')
print('=' * 80)
for d in compound_dirs[:5]:
    print(f'\n{os.path.basename(d)}')
    for fname in FILES_TO_CHECK:
        path = os.path.join(d, fname)
        exists = os.path.exists(path)
        if exists:
            size = os.path.getsize(path)
            if size > 1024**2:
                size_str = f'{size / 1024**2:.1f} MB'
            elif size > 1024:
                size_str = f'{size / 1024:.1f} KB'
            else:
                size_str = f'{size} B'
            print(f'  [✓] {fname:25s}  {size_str}')
        else:
            print(f'  [ ] {fname:25s}  --')
    incar_path = os.path.join(d, 'INCAR')
    print(f'  LWAVE setting: {check_lwave_in_incar(incar_path)}')

# Aggregate stats across ALL folders
print('\n' + '=' * 80)
print(f'AGGREGATE STATS across all {len(compound_dirs)} folders')
print('=' * 80)
counts = {f: 0 for f in FILES_TO_CHECK}
total_wavecar_size = 0
for d in compound_dirs:
    for fname in FILES_TO_CHECK:
        path = os.path.join(d, fname)
        if os.path.exists(path):
            counts[fname] += 1
            if fname == 'WAVECAR':
                total_wavecar_size += os.path.getsize(path)

for f, n in counts.items():
    pct = 100 * n / len(compound_dirs)
    print(f'  {f:25s}  {n:4d} / {len(compound_dirs)}  ({pct:5.1f}%)')

if counts['WAVECAR'] > 0:
    print(f'\n  Total WAVECAR storage: {total_wavecar_size / 1024**3:.2f} GB')

Found 99 compound folders

DETAILED CHECK: first 5 folders

AsBrS-1dcd471c2288
  [ ] POSCAR                     --
  [✓] POSCAR_std                 462 B
  [ ] CONTCAR                    --
  [ ] INCAR                      --
  [ ] POTCAR                     --
  [ ] KPOINTS                    --
  [ ] vasprun.xml                --
  [ ] WAVECAR                    --
  [ ] DOSCAR                     --
  [ ] CHGCAR                     --
  [ ] OUTCAR                     --
  [ ] lobsterin                  --
  [ ] COHPCAR.lobster            --
  [ ] ICOHPLIST.lobster          --
  LWAVE setting: INCAR missing

AsBrTe-671e6de2497a
  [ ] POSCAR                     --
  [✓] POSCAR_std                 462 B
  [ ] CONTCAR                    --
  [ ] INCAR                      --
  [ ] POTCAR                     --
  [ ] KPOINTS                    --
  [ ] vasprun.xml                --
  [ ] WAVECAR                    --
  [ ] DOSCAR                     --
  [ ] CHGCAR                     --

In [10]:
import os
import glob

POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')
compound_dirs = [d for d in glob.glob(os.path.join(POSCAR_DIR, '*'))
                 if os.path.isdir(d)]
print(f'Found {len(compound_dirs)} folders\n')

# STEP 1: actual ls of first 3 folders
print('=' * 80)
print('ACTUAL FILE LISTING (first 3 folders)')
print('=' * 80)
for d in compound_dirs[:3]:
    print(f'\n{os.path.basename(d)}/')
    for f in sorted(os.listdir(d)):
        path = os.path.join(d, f)
        size = os.path.getsize(path)
        if size > 1024**2:
            sz = f'{size/1024**2:.1f} MB'
        elif size > 1024:
            sz = f'{size/1024:.1f} KB'
        else:
            sz = f'{size} B'
        kind = 'DIR' if os.path.isdir(path) else 'file'
        print(f'  [{kind}] {f:40s}  {sz}')

Found 99 folders

ACTUAL FILE LISTING (first 3 folders)

AsBrS-1dcd471c2288/
  [file] POSCAR_std                                462 B
  [file] dos_As_dos.dat                            29.9 KB
  [file] dos_Br_dos.dat                            29.9 KB
  [file] dos_S_dos.dat                             29.9 KB
  [file] dos_dos.pdf                               23.2 KB
  [file] dos_total_dos.dat                         15.2 KB
  [file] ss_2d%2FAsBrS-1dcd471c2288%2Fbands_ncl%2FINCAR  412 B
  [file] ss_2d%2FAsBrS-1dcd471c2288%2Fbands_ncl%2FKPOINTS  6.7 KB
  [file] ss_2d%2FAsBrS-1dcd471c2288%2Fbands_ncl%2FOUTCAR  375.3 KB
  [file] ss_2d%2FAsBrS-1dcd471c2288%2Fbands_ncl%2FPOSCAR  440 B
  [file] ss_2d%2FAsBrS-1dcd471c2288%2Fbands_ncl%2Fvasprun.xml  10.2 MB
  [file] sumo-dosplot.log                          238 B

AsBrTe-671e6de2497a/
  [file] POSCAR_std                                462 B
  [file] dos_As_dos.dat                            29.9 KB
  [file] dos_Br_dos.dat                      

In [11]:
import os
import glob
import xml.etree.ElementTree as ET
from urllib.parse import unquote

POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')
compound_dirs = sorted([d for d in glob.glob(os.path.join(POSCAR_DIR, '*')) if os.path.isdir(d)])

def find_file_by_decoded_name(folder, target_decoded_name):
    """Look for a file whose URL-decoded name matches target."""
    for f in os.listdir(folder):
        if unquote(f).endswith(target_decoded_name):
            return os.path.join(folder, f)
    return None


def parse_vasprun_settings(vasprun_path):
    """Extract key INCAR settings from vasprun.xml."""
    if not vasprun_path or not os.path.exists(vasprun_path):
        return None
    try:
        tree = ET.parse(vasprun_path)
        root = tree.getroot()
        settings = {}
        # incar block
        for incar in root.iter('incar'):
            for item in incar:
                name = item.get('name')
                val = item.text.strip() if item.text else ''
                if name in ['LWAVE', 'LCHARG', 'ISYM', 'LSORBIT', 'LNONCOLLINEAR',
                            'ICHARG', 'LORBIT', 'NBANDS', 'ENCUT', 'NSW', 'IBRION']:
                    settings[name] = val
        # k-point info
        for kp in root.iter('kpoints'):
            for gen in kp.iter('generation'):
                settings['kpoint_scheme'] = gen.get('param')
                break
        # atom types from atominfo
        return settings
    except Exception as e:
        return {'_error': str(e)}


# Inventory
print('=' * 80)
print(f'INVENTORY across {len(compound_dirs)} folders')
print('=' * 80)

n_with_vasprun = 0
n_with_wavecar = 0
n_with_potcar = 0
n_with_chgcar = 0
n_with_outcar = 0
n_lwave_true = 0
n_lsorbit_true = 0
n_kpoint_path = 0
n_kpoint_mesh = 0

settings_summary = []
for d in compound_dirs:
    vasprun = find_file_by_decoded_name(d, '/vasprun.xml')
    wavecar = find_file_by_decoded_name(d, '/WAVECAR')
    potcar = find_file_by_decoded_name(d, '/POTCAR')
    chgcar = find_file_by_decoded_name(d, '/CHGCAR')
    outcar = find_file_by_decoded_name(d, '/OUTCAR')

    if vasprun: n_with_vasprun += 1
    if wavecar: n_with_wavecar += 1
    if potcar: n_with_potcar += 1
    if chgcar: n_with_chgcar += 1
    if outcar: n_with_outcar += 1

    if vasprun:
        s = parse_vasprun_settings(vasprun)
        if s and '_error' not in s:
            if s.get('LWAVE', '').strip().upper() in ['T', '.TRUE.', 'TRUE']:
                n_lwave_true += 1
            if s.get('LSORBIT', '').strip().upper() in ['T', '.TRUE.', 'TRUE']:
                n_lsorbit_true += 1
            scheme = s.get('kpoint_scheme', '')
            if scheme == 'listgenerated':
                n_kpoint_path += 1
            elif scheme in ['Monkhorst-Pack', 'Gamma']:
                n_kpoint_mesh += 1
            settings_summary.append({
                'uid': os.path.basename(d),
                'LWAVE': s.get('LWAVE', '?'),
                'LSORBIT': s.get('LSORBIT', '?'),
                'ISYM': s.get('ISYM', '?'),
                'ICHARG': s.get('ICHARG', '?'),
                'kpoints': scheme,
            })

print(f'\nFile presence:')
print(f'  vasprun.xml : {n_with_vasprun:3d} / {len(compound_dirs)}')
print(f'  WAVECAR     : {n_with_wavecar:3d} / {len(compound_dirs)}')
print(f'  POTCAR      : {n_with_potcar:3d} / {len(compound_dirs)}')
print(f'  CHGCAR      : {n_with_chgcar:3d} / {len(compound_dirs)}')
print(f'  OUTCAR      : {n_with_outcar:3d} / {len(compound_dirs)}')

print(f'\nVASP settings (from vasprun.xml):')
print(f'  LWAVE = .TRUE.   : {n_lwave_true:3d} / {n_with_vasprun} (need this for LOBSTER)')
print(f'  LSORBIT = .TRUE. : {n_lsorbit_true:3d} / {n_with_vasprun} (SOC turned on)')
print(f'  k-point: path    : {n_kpoint_path:3d} / {n_with_vasprun} (band structure run, NOT for LOBSTER)')
print(f'  k-point: MP mesh : {n_kpoint_mesh:3d} / {n_with_vasprun} (good for LOBSTER)')

print('\nFirst 5 compounds — settings:')
import pandas as pd
print(pd.DataFrame(settings_summary[:5]).to_string(index=False))

INVENTORY across 99 folders

File presence:
  vasprun.xml :  99 / 99
  WAVECAR     :   0 / 99
  POTCAR      :   0 / 99
  CHGCAR      :   0 / 99
  OUTCAR      :  99 / 99

VASP settings (from vasprun.xml):
  LWAVE = .TRUE.   :  99 / 99 (need this for LOBSTER)
  LSORBIT = .TRUE. :  99 / 99 (SOC turned on)
  k-point: path    :   0 / 99 (band structure run, NOT for LOBSTER)
  k-point: MP mesh :   0 / 99 (good for LOBSTER)

First 5 compounds — settings:
                uid LWAVE LSORBIT ISYM ICHARG kpoints
 AsBrS-1dcd471c2288     T       T    ?     11        
AsBrTe-671e6de2497a     T       T    ?     11        
AsClSe-1a3be826b3e0     T       T    ?     11        
AsClTe-4fd8ad708fb0     T       T    ?     11        
AsClTe-fba4cc0df459     T       T    ?     11        


In [12]:
# Quick re-check: what does the kpoint generation parameter actually say?
import xml.etree.ElementTree as ET
from urllib.parse import unquote

d = compound_dirs[0]
vasprun = None
for f in os.listdir(d):
    if unquote(f).endswith('/vasprun.xml'):
        vasprun = os.path.join(d, f)
        break

tree = ET.parse(vasprun)
root = tree.getroot()
for kp in root.iter('kpoints'):
    print('Inside <kpoints>:')
    for child in kp:
        print(f'  <{child.tag}>', dict(child.attrib))
        if child.tag == 'generation':
            for gc in child:
                print(f'    <{gc.tag}>', dict(gc.attrib), '=', gc.text.strip() if gc.text else '')

# Count number of k-points actually in the file
n_kpoints = 0
for kp in root.iter('kpoints'):
    for vlist in kp.iter('varray'):
        if vlist.get('name') == 'kpointlist':
            n_kpoints = len(list(vlist))
            break
    break
print(f'\nNumber of k-points in vasprun.xml: {n_kpoints}')
print('  (small ~30-100 = path; large ~64-1000+ = mesh)')

Inside <kpoints>:
  <varray> {'name': 'kpointlist'}
  <varray> {'name': 'weights'}

Number of k-points in vasprun.xml: 213
  (small ~30-100 = path; large ~64-1000+ = mesh)


In [13]:
# Look at the actual k-point coordinates to see if they're a path or scattered mesh
n_kp = 213
kpoints = []
for kp in root.iter('kpoints'):
    for vlist in kp.iter('varray'):
        if vlist.get('name') == 'kpointlist':
            for v in vlist:
                kpoints.append([float(x) for x in v.text.split()])
            break
    break

import numpy as np
kpoints = np.array(kpoints)
print(f'Total k-points: {len(kpoints)}')
print(f'\nFirst 5 k-points:')
print(kpoints[:5])
print(f'\nLast 5 k-points:')
print(kpoints[-5:])

# Distance between consecutive k-points
diffs = np.linalg.norm(np.diff(kpoints, axis=0), axis=1)
print(f'\nDistances between consecutive k-points:')
print(f'  mean: {diffs.mean():.4f}')
print(f'  std:  {diffs.std():.4f}')
print(f'  max:  {diffs.max():.4f}')
print(f'  min:  {diffs.min():.4f}')

# If it's a path: small consistent gaps, occasional jumps at segment boundaries
# If it's a mesh: scattered, no consistent ordering, large mean distance

Total k-points: 213

First 5 k-points:
[[0.       0.       0.      ]
 [0.006494 0.       0.      ]
 [0.012987 0.       0.      ]
 [0.019481 0.       0.      ]
 [0.025974 0.       0.      ]]

Last 5 k-points:
[[0.014652 0.014652 0.      ]
 [0.010989 0.010989 0.      ]
 [0.007326 0.007326 0.      ]
 [0.003663 0.003663 0.      ]
 [0.       0.       0.      ]]

Distances between consecutive k-points:
  mean: 0.0063
  std:  0.0012
  max:  0.0085
  min:  0.0052


In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
from time import time
from itertools import combinations

from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join('..'))
OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')
POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')
RESULTS_DIR = os.path.join('.', 'nb13_symmetry-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup OK. Results dir:', RESULTS_DIR)


Setup OK. Results dir: .\nb13_symmetry-results


## Cell 2: Load merged data, baseline (same as nb9, nb10, nb12)

In [2]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)
ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
for col in old_features:
    df_merged[f'old_{col}' if col in overlap else col] = df_old[col].values
for col in new_features:
    df_merged[f'new_{col}' if col in overlap else col] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values

XGB_REG_PARAMS = dict(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0,
)
BASELINE_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']

def resolve(name, cols):
    if name in cols: return name
    if f'old_{name}' in cols: return f'old_{name}'
    if f'new_{name}' in cols: return f'new_{name}'
    raise KeyError(name)

BASELINE = [resolve(n, df_99.columns) for n in BASELINE_RAW]

def eval_reg_99(features, df, y):
    X = df[features].fillna(0).values
    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return r2_score(y, y_pred), mean_absolute_error(y, y_pred)

r2_base, mae_base = eval_reg_99(BASELINE, df_99, y_99)
print(f'Baseline LOO R2 = {r2_base:.4f}, MAE = {mae_base:.4f}')
if abs(r2_base - 0.643) > 0.05:
    print('  WARNING: baseline differs from nb7 by > 0.05')


Baseline LOO R2 = 0.6434, MAE = 0.4287


## Cell 3: Extract symmetry descriptors per compound

For each POSCAR_std, use `SpacegroupAnalyzer` to get:

| Feature | Meaning | Range |
|---|---|---|
| `sym_spacegroup_number` | International Tables number (1-230) | int |
| `sym_crystal_system_id` | 1=triclinic, 2=monoclinic, 3=orthorhombic, 4=tetragonal, 5=trigonal, 6=hexagonal, 7=cubic | int 1-7 |
| `sym_is_centrosymmetric` | Has inversion center (1=yes, 0=no) | binary |
| `sym_n_symmetry_ops` | Number of symmetry operations (order of point group) | int 1-48 |
| `sym_n_unique_sites` | Number of symmetrically inequivalent atomic sites | int |
| `sym_n_atoms_total` | Total atoms in unit cell (for context) | int |
| `sym_unique_per_total` | Ratio: unique sites / total atoms (high = low symmetry) | 0-1 |
| `sym_n_planes_polar_axis` | Number of mirror planes containing polar axis (relevant for 2D Rashba) | int |
| `sym_has_polar_axis` | Whether structure has a unique polar axis (1=yes) | binary |

**Spglib symprec:** the tolerance for considering atoms as equivalent. Default 0.01 Å is good for DFT-relaxed structures. We use 0.1 Å to be slightly looser since C2DB structures may have small numerical asymmetries.

Time: ~30-60 sec for 99 compounds.


In [3]:
POLAR_POINTGROUPS = {'1', '2', 'm', 'mm2', '4', '4mm', '3', '3m', '6', '6mm'}

CRYSTAL_SYSTEM_ID = {
    'triclinic': 1, 'monoclinic': 2, 'orthorhombic': 3,
    'tetragonal': 4, 'trigonal': 5, 'hexagonal': 6, 'cubic': 7,
}


def load_structure(uid):
    matches = glob.glob(os.path.join(POSCAR_DIR, f'*-{uid}', 'POSCAR_std'))
    if not matches:
        return None
    try:
        return Structure.from_file(matches[0])
    except Exception:
        return None


def extract_symmetry(struct, symprec=0.1):
    if struct is None:
        return None
    try:
        sga = SpacegroupAnalyzer(struct, symprec=symprec)
        sg_num = sga.get_space_group_number()
        sg_symbol = sga.get_space_group_symbol()
        crystal_sys = sga.get_crystal_system()
        point_group = sga.get_point_group_symbol()
        sym_ops = sga.get_symmetry_operations()
        symm_struct = sga.get_symmetrized_structure()
        n_unique_sites = len(symm_struct.equivalent_indices)

        # Centrosymmetric? Check for inversion in the symmetry ops
        is_centro = 0
        for op in sym_ops:
            mat = op.rotation_matrix
            # Inversion: rotation = -I
            if np.allclose(mat, -np.eye(3), atol=1e-3):
                is_centro = 1
                break

        # Polar point group?
        has_polar = 1 if point_group in POLAR_POINTGROUPS else 0

        # Mirror planes: rotation matrix has det = -1 (improper) and is a reflection (mat^2 = I)
        n_mirror_planes = 0
        for op in sym_ops:
            mat = op.rotation_matrix
            if np.isclose(np.linalg.det(mat), -1.0, atol=1e-3):
                if np.allclose(mat @ mat, np.eye(3), atol=1e-3):
                    n_mirror_planes += 1

        return {
            'sym_spacegroup_number': int(sg_num),
            'sym_crystal_system_id': CRYSTAL_SYSTEM_ID.get(crystal_sys, 0),
            'sym_is_centrosymmetric': int(is_centro),
            'sym_n_symmetry_ops': len(sym_ops),
            'sym_n_unique_sites': int(n_unique_sites),
            'sym_n_atoms_total': len(struct),
            'sym_unique_per_total': float(n_unique_sites) / len(struct),
            'sym_n_mirror_planes': int(n_mirror_planes),
            'sym_has_polar_axis': int(has_polar),
            'sym_pointgroup_str': point_group,
            'sym_spacegroup_symbol': sg_symbol,
        }
    except Exception as e:
        return {'_error': str(e)}


print('Extracting symmetry features for 99 compounds...')
sym_rows = []
n_failed = 0
t0 = time()

for i, row in df_99.iterrows():
    uid = row['uid']
    struct = load_structure(uid)
    sym = extract_symmetry(struct)
    if sym is None or '_error' in (sym or {}):
        n_failed += 1
        sym = {k: np.nan for k in ['sym_spacegroup_number', 'sym_crystal_system_id',
            'sym_is_centrosymmetric', 'sym_n_symmetry_ops', 'sym_n_unique_sites',
            'sym_n_atoms_total', 'sym_unique_per_total', 'sym_n_mirror_planes',
            'sym_has_polar_axis']}
        sym['sym_pointgroup_str'] = 'NA'
        sym['sym_spacegroup_symbol'] = 'NA'
    sym['uid'] = uid
    sym_rows.append(sym)
    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/99] elapsed {time()-t0:.0f}s, failed {n_failed}')

sym_df = pd.DataFrame(sym_rows)
print(f'\nDone in {time()-t0:.0f}s. Failed: {n_failed}/99')
print(f'sym_df shape: {sym_df.shape}')
print('\nSample:')
print(sym_df.head())
sym_df.to_csv(os.path.join(RESULTS_DIR, 'symmetry_features.csv'), index=False)


Extracting symmetry features for 99 compounds...
  [20/99] elapsed 0s, failed 0
  [40/99] elapsed 0s, failed 0
  [60/99] elapsed 0s, failed 0
  [80/99] elapsed 1s, failed 0

Done in 1s. Failed: 0/99
sym_df shape: (99, 12)

Sample:
   sym_spacegroup_number  sym_crystal_system_id  sym_is_centrosymmetric  \
0                    156                      5                       0   
1                     31                      3                       0   
2                    156                      5                       0   
3                     25                      3                       0   
4                     25                      3                       0   

   sym_n_symmetry_ops  sym_n_unique_sites  sym_n_atoms_total  \
0                   6                   3                  3   
1                   4                   2                  4   
2                   6                   3                  3   
3                   4                   6                 12  

## Cell 4: Quick sanity check — does symmetry correlate with alpha_R?

Before any ML, do the simple thing: plot/print alpha_R distribution by space group, by centrosymmetry flag, by crystal system. If centrosymmetric compounds in your dataset have α_R near zero (as they should by symmetry), and non-centro have a wide spread, that's the expected pattern.

In [4]:
# Merge symmetry features with df_99 for analysis
df_99_ext = df_99.merge(sym_df, on='uid', how='left')

print('Centrosymmetric flag distribution:')
print(df_99_ext.groupby('sym_is_centrosymmetric').agg(
    n=('uid', 'count'),
    alpha_mean=(TARGET, 'mean'),
    alpha_median=(TARGET, 'median'),
    alpha_std=(TARGET, 'std'),
    alpha_max=(TARGET, 'max'),
).round(3))

print('\nCrystal system distribution:')
print(df_99_ext.groupby('sym_crystal_system_id').agg(
    n=('uid', 'count'),
    alpha_mean=(TARGET, 'mean'),
    alpha_max=(TARGET, 'max'),
).round(3))

print('\nTop 10 most common space groups in dataset:')
print(df_99_ext['sym_spacegroup_symbol'].value_counts().head(10))

# Correlation of numeric symmetry features with alpha_R
sym_num_cols = ['sym_spacegroup_number', 'sym_crystal_system_id', 'sym_is_centrosymmetric',
                'sym_n_symmetry_ops', 'sym_n_unique_sites', 'sym_unique_per_total',
                'sym_n_mirror_planes', 'sym_has_polar_axis']
print('\nCorrelation with Rashba parameter:')
for col in sym_num_cols:
    if col in df_99_ext.columns:
        c = df_99_ext[col].corr(df_99_ext[TARGET])
        print(f'  {col:35s}  r = {c:+.3f}')


Centrosymmetric flag distribution:
                         n  alpha_mean  alpha_median  alpha_std  alpha_max
sym_is_centrosymmetric                                                    
0                       99       1.768         1.501      0.961      4.804

Crystal system distribution:
                        n  alpha_mean  alpha_max
sym_crystal_system_id                           
1                       2       1.178      1.406
2                       6       1.441      3.064
3                      36       1.694      4.804
5                      55       1.874      4.457

Top 10 most common space groups in dataset:
sym_spacegroup_symbol
P3m1      50
Pmm2      31
P31m       4
Pm         3
Pma2       3
Pmn2_1     2
P1         2
P2         2
P2_1       1
P3         1
Name: count, dtype: int64

Correlation with Rashba parameter:
  sym_spacegroup_number                r = +0.132
  sym_crystal_system_id                r = +0.146
  sym_is_centrosymmetric               r = +nan
  sym_n_s

## Cell 5: Phase A — C6 + 1

Take only numeric symmetry columns as candidates. Run C6+1 LOO eval.

In [5]:
SYM_CANDIDATES = ['sym_spacegroup_number', 'sym_crystal_system_id', 'sym_is_centrosymmetric',
                  'sym_n_symmetry_ops', 'sym_n_unique_sites', 'sym_n_atoms_total',
                  'sym_unique_per_total', 'sym_n_mirror_planes', 'sym_has_polar_axis']

# Drop any that aren't in df_99_ext (defensive)
SYM_CANDIDATES = [c for c in SYM_CANDIDATES if c in df_99_ext.columns]

# Drop constant columns
nuniq = df_99_ext[SYM_CANDIDATES].nunique()
SYM_CANDIDATES = [c for c in SYM_CANDIDATES if nuniq[c] > 1]
print(f'Active symmetry candidates: {len(SYM_CANDIDATES)}')
print(f'  {SYM_CANDIDATES}\n')

print('PHASE A: C6 + 1')
phase_a = []
for cand in SYM_CANDIDATES:
    feats = BASELINE + [cand]
    r2, mae = eval_reg_99(feats, df_99_ext, y_99)
    phase_a.append({'candidate': cand, 'r2': r2, 'mae': mae,
                    'delta_r2': r2 - r2_base})

phase_a_df = pd.DataFrame(phase_a).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print(phase_a_df.to_string(index=False))
phase_a_df.to_csv(os.path.join(RESULTS_DIR, 'phase_a_symmetry.csv'), index=False)

print(f'\nBaseline: {r2_base:.4f}')
print(f'Best C6+1: {phase_a_df.iloc[0]["r2"]:.4f} ({phase_a_df.iloc[0]["delta_r2"]:+.4f})')


Active symmetry candidates: 7
  ['sym_spacegroup_number', 'sym_crystal_system_id', 'sym_n_symmetry_ops', 'sym_n_unique_sites', 'sym_n_atoms_total', 'sym_unique_per_total', 'sym_n_mirror_planes']

PHASE A: C6 + 1
            candidate       r2      mae  delta_r2
sym_spacegroup_number 0.619636 0.421662 -0.023746
   sym_n_symmetry_ops 0.615275 0.425688 -0.028108
  sym_n_mirror_planes 0.612576 0.430605 -0.030806
sym_crystal_system_id 0.612468 0.430209 -0.030915
    sym_n_atoms_total 0.609161 0.430160 -0.034222
 sym_unique_per_total 0.604043 0.432752 -0.039340
   sym_n_unique_sites 0.601196 0.431602 -0.042187

Baseline: 0.6434
Best C6+1: 0.6196 (-0.0237)


## Cell 6: Phase B — C6 + 2 from top features

Take top 5 from Phase A, evaluate all 10 pairs. (With only ~9 candidates total, "top 5" is most of them.)

In [6]:
K = min(5, len(phase_a_df))
top_k = phase_a_df.head(K)['candidate'].tolist()
print(f'Top {K} from Phase A: {top_k}\n')

phase_b = []
for c1, c2 in combinations(top_k, 2):
    feats = BASELINE + [c1, c2]
    r2, mae = eval_reg_99(feats, df_99_ext, y_99)
    phase_b.append({'cand_1': c1, 'cand_2': c2, 'r2': r2, 'mae': mae,
                    'delta_r2': r2 - r2_base})

phase_b_df = pd.DataFrame(phase_b).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print('All C6+2 pairs:')
print(phase_b_df.to_string(index=False))
phase_b_df.to_csv(os.path.join(RESULTS_DIR, 'phase_b_symmetry.csv'), index=False)

print(f'\nBaseline:   {r2_base:.4f}')
print(f'Best C6+1:  {phase_a_df.iloc[0]["r2"]:.4f}')
print(f'Best C6+2:  {phase_b_df.iloc[0]["r2"]:.4f}')


Top 5 from Phase A: ['sym_spacegroup_number', 'sym_n_symmetry_ops', 'sym_n_mirror_planes', 'sym_crystal_system_id', 'sym_n_atoms_total']

All C6+2 pairs:
               cand_1                cand_2       r2      mae  delta_r2
sym_spacegroup_number sym_crystal_system_id 0.599898 0.444405 -0.043484
sym_spacegroup_number    sym_n_symmetry_ops 0.599896 0.444690 -0.043487
sym_crystal_system_id     sym_n_atoms_total 0.586451 0.446001 -0.056932
sym_spacegroup_number   sym_n_mirror_planes 0.580621 0.453065 -0.062761
  sym_n_mirror_planes     sym_n_atoms_total 0.580241 0.450533 -0.063142
  sym_n_mirror_planes sym_crystal_system_id 0.579629 0.451617 -0.063754
   sym_n_symmetry_ops     sym_n_atoms_total 0.577748 0.451105 -0.065635
sym_spacegroup_number     sym_n_atoms_total 0.575164 0.454085 -0.068219
   sym_n_symmetry_ops   sym_n_mirror_planes 0.573903 0.462851 -0.069479
   sym_n_symmetry_ops sym_crystal_system_id 0.573154 0.461850 -0.070229

Baseline:   0.6434
Best C6+1:  0.6196
Best C6+2:  0.5

## Cell 7: Phase C — C6 + 3 (only if Phase B helped)

In [7]:
if phase_b_df.iloc[0]['r2'] > phase_a_df.iloc[0]['r2']:
    phase_c = []
    for c1, c2, c3 in combinations(top_k, 3):
        feats = BASELINE + [c1, c2, c3]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_c.append({'cand_1': c1, 'cand_2': c2, 'cand_3': c3,
                        'r2': r2, 'mae': mae, 'delta_r2': r2 - r2_base})
    phase_c_df = pd.DataFrame(phase_c).sort_values('delta_r2', ascending=False).reset_index(drop=True)
    print('Phase C (C6 + 3):')
    print(phase_c_df.head(10).to_string(index=False))
    phase_c_df.to_csv(os.path.join(RESULTS_DIR, 'phase_c_symmetry.csv'), index=False)
    print(f'\nBest C6+3: {phase_c_df.iloc[0]["r2"]:.4f}')
else:
    print('Phase B did not improve over Phase A. Skipping C.')
    phase_c_df = pd.DataFrame()


Phase B did not improve over Phase A. Skipping C.


## Cell 8: Multi-seed validation on the best new feature

If we found something that looks like a real gain, validate with 10 seeds. If gain is < 1 std above baseline's std, it's noise.

In [8]:
# Pick the best result
best_extras = []
best_r2 = r2_base
for src_df, src_name in [(phase_a_df, 'A'), (phase_b_df, 'B'), (phase_c_df, 'C') if len(phase_c_df) else (None, None)]:
    if src_df is None or len(src_df) == 0: continue
    if src_df.iloc[0]['r2'] > best_r2:
        best_r2 = src_df.iloc[0]['r2']
        if src_name == 'A':
            best_extras = [src_df.iloc[0]['candidate']]
        elif src_name == 'B':
            best_extras = [src_df.iloc[0]['cand_1'], src_df.iloc[0]['cand_2']]
        else:
            best_extras = [src_df.iloc[0]['cand_1'], src_df.iloc[0]['cand_2'], src_df.iloc[0]['cand_3']]

if not best_extras or best_r2 <= r2_base:
    print('No symmetry features helped over baseline. Reporting the negative result.')
else:
    print(f'Best feature set: C6 + {best_extras}')
    print(f'  R2 (single seed): {best_r2:.4f}')
    print(f'  Validating with 10 seeds...')

    new_feats = BASELINE + best_extras
    r2s_new, r2s_base_seeds = [], []
    for seed in range(10):
        params = dict(XGB_REG_PARAMS); params['random_state'] = seed
        m = XGBRegressor(**params)
        X_new = df_99_ext[new_feats].fillna(0).values
        X_base = df_99_ext[BASELINE].fillna(0).values
        y_pred_new = cross_val_predict(m, X_new, y_99, cv=LeaveOneOut())
        m2 = XGBRegressor(**params)
        y_pred_base = cross_val_predict(m2, X_base, y_99, cv=LeaveOneOut())
        r2s_new.append(r2_score(y_99, y_pred_new))
        r2s_base_seeds.append(r2_score(y_99, y_pred_base))

    print(f'\n10-seed C6 baseline:  R2 = {np.mean(r2s_base_seeds):.4f} +/- {np.std(r2s_base_seeds):.4f}')
    print(f'10-seed with sym:     R2 = {np.mean(r2s_new):.4f} +/- {np.std(r2s_new):.4f}')
    diff = np.mean(r2s_new) - np.mean(r2s_base_seeds)
    pooled_std = np.sqrt(np.std(r2s_new)**2 + np.std(r2s_base_seeds)**2)
    print(f'Difference: {diff:+.4f}, ratio to pooled std: {diff/pooled_std if pooled_std > 0 else 0:.2f}')
    print('  ratio > 2: real signal')
    print('  ratio 1-2: marginal')
    print('  ratio < 1: noise')


No symmetry features helped over baseline. Reporting the negative result.


## Now stop.

Whatever this notebook produces, you write it down (positive, negative, or marginal) and stop touching the notebook tonight. Sleep. DSA tomorrow morning before anything else.

If symmetry features give +0.02 or more with ratio > 2 in the multi-seed test, that's a real finding to add to your report.
If they give nothing, that's also a real finding — you've shown that the existing C6 features already implicitly encode the symmetry information that matters (probably via the orbital fractions and k-path angle).

Either way, you have your DDP story:

> "We tested four descriptor expansions: elemental aggregations (no significant gain), bond-weighted distribution function proxy (no significant gain), SISSO-style symbolic regression (interpretable formula at slightly lower R²), and crystallographic symmetry features (result here). Combined with a residual analysis identifying the heavy-chalcogenide regime as out-of-domain, the C6 model achieves R² = 0.643 on 99 2D materials. Future work targets ICOHP-based bonding descriptors and Janus-asymmetry features to address the polymorph indistinguishability case (BiITe at α_R = 0.47 vs 2.09 for two crystal structures)."

That's a complete chapter. You're done.

Get rest.
